In [1]:
import pandas as pd

In [13]:
from pathlib import Path

import numpy as np
import optuna
from catboost import CatBoostClassifier
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Kaggle: Settings → Accelerator → GPU (P100/T4) for CatBoost CUDA below.
# sklearn HistGradientBoosting + Optuna + ColumnTransformer stay CPU-only (expected).
# TPUs (e.g. v5e) are for JAX/TF stacks; they do not run sklearn/CatBoost — pick GPU, not TPU, here.


def catboost_accelerator_kwargs() -> dict:
    """Use CatBoost on GPU when a CUDA device is available (Kaggle GPU notebook or local CUDA)."""
    try:
        from catboost.utils import get_gpu_device_count

        if get_gpu_device_count() > 0:
            return {"task_type": "GPU", "devices": "0"}
    except Exception:
        pass
    return {"task_type": "CPU"}


def find_train_csv() -> Path:
    cwd = Path.cwd()
    for root in (cwd, cwd.parent):
        p = root / "input" / "train.csv"
        if p.exists():
            return p
    raise FileNotFoundError(
        f"Could not find input/train.csv from cwd or parent. cwd={cwd!s}"
    )


TRAIN_CSV = find_train_csv()
print("Using:", TRAIN_CSV.resolve())
print("CatBoost accelerator:", catboost_accelerator_kwargs())

# Filled by the Optuna cell when you run it; `{}` keeps the default clf hyperparameters below.
CLF_BEST_KWARGS: dict = {}

Using: /home/nabin2004/Desktop/SwissKit/input/train.csv


In [6]:
df = pd.read_csv(TRAIN_CSV)
TARGET = "Irrigation_Need"
ID_COL = "id"
print(df.shape)
print(df.dtypes)
print(df[TARGET].value_counts())
print("Missing per column:\n", df.isna().sum().sort_values(ascending=False).head(20))
df.head()

(630000, 21)
id                           int64
Soil_Type                   object
Soil_pH                    float64
Soil_Moisture              float64
Organic_Carbon             float64
Electrical_Conductivity    float64
Temperature_C              float64
Humidity                   float64
Rainfall_mm                float64
Sunlight_Hours             float64
Wind_Speed_kmh             float64
Crop_Type                   object
Crop_Growth_Stage           object
Season                      object
Irrigation_Type             object
Water_Source                object
Field_Area_hectare         float64
Mulching_Used               object
Previous_Irrigation_mm     float64
Region                      object
Irrigation_Need             object
dtype: object
Irrigation_Need
Low       369917
Medium    239074
High       21009
Name: count, dtype: int64
Missing per column:
 id                         0
Soil_Type                  0
Soil_pH                    0
Soil_Moisture              0
Organic_

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


In [7]:
# Drop id from features; keep target separate
feature_cols = [c for c in df.columns if c not in (ID_COL, TARGET)]

# Adjust lists if your file differs
numeric_features = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
categorical_features = [c for c in feature_cols if c not in numeric_features]

X = df[feature_cols]
y = df[TARGET]

print("Numeric:", numeric_features)
print("Categorical:", categorical_features)

Numeric: ['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']
Categorical: ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']


In [ ]:
# Optuna: tune HistGradientBoosting on a subset; objective = validation balanced_accuracy_score.
# Re-run after changing OPTUNA_TRIALS / OPTUNA_SAMPLE. Then re-run the pipeline + training cells.

OPTUNA_SAMPLE = min(120_000, len(X))
OPTUNA_TRIALS = 25

rng = np.random.default_rng(0)
idx = rng.choice(len(X), size=OPTUNA_SAMPLE, replace=False)
X_t, y_t = X.iloc[idx], y.iloc[idx]
X_tr, X_va, y_tr, y_va = train_test_split(
    X_t, y_t, test_size=0.2, random_state=0, stratify=y_t
)


def _make_prep() -> ColumnTransformer:
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric_features),
            (
                "cat",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                categorical_features,
            ),
        ],
    )


def _objective(trial: optuna.Trial) -> float:
    clf = HistGradientBoostingClassifier(
        random_state=42,
        class_weight="balanced",
        learning_rate=trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
        max_iter=trial.suggest_int("max_iter", 80, 400),
        max_depth=trial.suggest_int("max_depth", 4, 16),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 10, 200),
        l2_regularization=trial.suggest_float("l2_regularization", 1e-5, 1.0, log=True),
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=15,
    )
    est = Pipeline([("prep", _make_prep()), ("clf", clf)])
    est.fit(X_tr, y_tr)
    pred = est.predict(X_va)
    return float(balanced_accuracy_score(y_va, pred))


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=0),
)
study.optimize(_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)

CLF_BEST_KWARGS = dict(study.best_params)
print("Best validation balanced accuracy:", study.best_value)
print("CLF_BEST_KWARGS (used in next cell):", CLF_BEST_KWARGS)

In [8]:
preprocess = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features,
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_features,
        ),
    ],
)

# Defaults; overwritten by keys from CLF_BEST_KWARGS after running the Optuna cell.
clf_kw = dict(
    random_state=42,
    class_weight="balanced",
    learning_rate=0.08,
    max_iter=200,
    max_depth=8,
    early_stopping=True,
    validation_fraction=0.05,
    n_iter_no_change=15,
)
clf_kw.update(CLF_BEST_KWARGS)

# HistGradientBoosting handles large tabular data well; RandomForest is a simple alternative
clf = HistGradientBoostingClassifier(**clf_kw)

model = Pipeline(
    steps=[
        ("prep", preprocess),
        ("clf", clf),
    ]
)

In [9]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

model.fit(X_train, y_train)
y_pred = model.predict(X_val)

print("Balanced accuracy (val):", balanced_accuracy_score(y_val, y_pred))
print(classification_report(y_val, y_pred))
print("Confusion matrix (rows=true, cols=pred):\n", confusion_matrix(y_val, y_pred))

              precision    recall  f1-score   support

        High       0.88      0.95      0.92      4202
         Low       0.99      0.99      0.99     73983
      Medium       0.99      0.97      0.98     47815

    accuracy                           0.98    126000
   macro avg       0.95      0.97      0.96    126000
weighted avg       0.98      0.98      0.98    126000

Confusion matrix (rows=true, cols=pred):
 [[ 4006     0   196]
 [    0 73511   472]
 [  532  1007 46276]]


In [ ]:
# CatBoost: uses raw categoricals (no one-hot). Same train/val split as the sklearn `model` above.
cat_clf = CatBoostClassifier(
    iterations=1200,
    learning_rate=0.05,
    depth=8,
    loss_function="MultiClass",
    eval_metric="TotalF1:average=Macro",
    random_seed=42,
    verbose=200,
    early_stopping_rounds=80,
    auto_class_weights="Balanced",
    **catboost_accelerator_kwargs(),
)

cat_clf.fit(
    X_train,
    y_train,
    cat_features=list(categorical_features),
    eval_set=(X_val, y_val),
    use_best_model=True,
)

y_pred_cb = np.asarray(cat_clf.predict(X_val)).ravel()
print("CatBoost balanced accuracy (val):", balanced_accuracy_score(y_val, y_pred_cb))
print("CatBoost (val):\n", classification_report(y_val, y_pred_cb, digits=4))

In [ ]:
# Soft ensemble: average predict_proba after aligning CatBoost columns to sklearn `model.classes_`.
classes_sk = np.asarray(model.classes_)
p_sk = model.predict_proba(X_val)

cb_classes = np.asarray(cat_clf.classes_)
p_cb = cat_clf.predict_proba(X_val)
col_order = [int(np.where(cb_classes == c)[0][0]) for c in classes_sk]
p_cb_aligned = p_cb[:, col_order]

blend_w = 0.5  # CatBoost weight in [0, 1]; tune on validation
p_ens = (1.0 - blend_w) * p_sk + blend_w * p_cb_aligned
y_pred_ens = classes_sk[np.argmax(p_ens, axis=1)]

print(f"Ensemble sklearn={1 - blend_w:.2f}, catboost={blend_w:.2f} (val):\n")
print("Ensemble balanced accuracy (val):", balanced_accuracy_score(y_val, y_pred_ens))
print(classification_report(y_val, y_pred_ens, digits=4))

In [10]:
# Use a subset for faster CV during notebook exploration
SAMPLE_N = 50_000
if len(df) > SAMPLE_N:
    rng = np.random.default_rng(42)
    idx = rng.choice(len(df), size=SAMPLE_N, replace=False)
    X_cv, y_cv = X.iloc[idx], y.iloc[idx]
else:
    X_cv, y_cv = X, y

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
scores = cross_val_score(
    model,
    X_cv,
    y_cv,
    cv=cv,
    scoring="balanced_accuracy",
    n_jobs=-1,
)
print(
    "Balanced accuracy (3-fold, possibly subsampled):",
    scores,
    "mean:",
    scores.mean(),
)

F1 macro (3-fold, possibly subsampled): [0.96003982 0.95833625 0.95495524] mean: 0.9577771036868005


In [11]:
model.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('prep', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains spa

In [ ]:
# CatBoost on full `X`, `y` (run the CatBoost validation cell first so `cat_clf` exists).
_best_it = cat_clf.get_best_iteration()
_n_est = int(_best_it) + 150 if _best_it is not None and _best_it > 0 else 1200

cat_full = CatBoostClassifier(
    iterations=_n_est,
    learning_rate=0.05,
    depth=8,
    loss_function="MultiClass",
    random_seed=42,
    verbose=200,
    auto_class_weights="Balanced",
    **catboost_accelerator_kwargs(),
)
cat_full.fit(X, y, cat_features=list(categorical_features))

ARTIFACT_CB = Path("../models/irrigation_catboost.cbm")
ARTIFACT_CB.parent.mkdir(parents=True, exist_ok=True)
cat_full.save_model(str(ARTIFACT_CB))
print("Saved CatBoost:", ARTIFACT_CB.resolve())

In [12]:
from joblib import dump

ARTIFACT = Path("../models/irrigation_need_model.joblib")
dump(model, ARTIFACT)
print("Saved:", ARTIFACT.resolve())
# loaded = load(ARTIFACT)

Saved: /home/nabin2004/Desktop/SwissKit/models/irrigation_need_model.joblib
